# Generate System Level Synthetic Housing and Homelessness Administrative (SHHAd) Data

```
Program to export synthetic housing and homelessness system of care data in HIFIS table format.
Copyright (C) 2026 Geoffrey Messier

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program. If not, see <https://www.gnu.org/licenses/>. 
```

In [1]:
%load_ext autoreload
%autoreload 1

In [2]:
import numpy as np
import pandas as pd
import re, copy
import pickle

from tqdm.auto import tqdm, trange
from tqdm.notebook import tqdm
tqdm.pandas()

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib as mpl

from scipy.special import factorial
import scipy.stats as stats

import sys
sys.path.insert(0, '../util/')

In [3]:
COLOR = 'black'
mpl.rcParams['text.color'] = COLOR
mpl.rcParams['axes.labelcolor'] = COLOR
mpl.rcParams['xtick.color'] = COLOR
mpl.rcParams['ytick.color'] = COLOR

In [4]:
procDirStr = './data/'

In [5]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 500)
np.set_printoptions(threshold=200)
np.set_printoptions(linewidth=200)
np.set_printoptions(threshold=sys.maxsize)

In [6]:
np.random.seed(35461432)

In [7]:
def dump_pickle(x,fileStr):
    with open(fileStr, 'wb') as f:
        pickle.dump(x, f, pickle.HIGHEST_PROTOCOL)        
        
def read_pickle(fileStr):
    with open(fileStr, 'rb') as f:
        return pickle.load(f)

## Generate Markov Model
---

In [8]:
def calc_q_shelter_housing(x,converged,nS,nH,wAvgS,wAvgH,alphaDiv,alphaTSHpreD,alphaTSHpstD,loadTrg):

    # Dwell times.
    wsS = x[:nS*2]*wAvgS  # Separate rates for pre/pst diversion shelter.
    wsH = x[nS*2:nS*2+nH]*wAvgH
    
    # Arrival rates.
    lam = np.zeros(nS*4+nH) 
    lam[:2*nS] = x[ nS*2+nH : nS*2+nH + 2*nS ]
    # Arrival rates for the nS postDiv shelter and nH housing programs are zero.
    
    alphaR = x[-4]
    alphaTHS = x[-3]    
    alphaXS = x[-2]
    alphaXH = x[-1]
    
    Q = np.zeros((nS*4+nH+1,nS*4+nH+1))
    
    # Shelter Programs (preDiversion)
    for k in range(nS):
        # Absent but returning to shelter.        
        Q[2*k,2*k+1] = alphaR / wsS[k] * (1-alphaTSHpreD-alphaDiv-alphaXS) / (1+alphaR)
    
        # Transition to other shelter.  Split between higher and lower shelter numbers.
        Q[2*k,(2*k+2) % (2*nS)] = 0.5 / wsS[k] * (1-alphaTSHpreD-alphaDiv-alphaXS) / (1+alphaR)  # Higher
        Q[2*k,(2*k+2*(nS-1)) % (2*nS)] = 0.5 / wsS[k] * (1-alphaTSHpreD-alphaDiv-alphaXS) / (1+alphaR)  # Higher
        
        # System exit.
        Q[2*k,nS*4+nH] = alphaXS / wsS[k]  
    
        # Transition to housing.  Split equally across all housing programs (for now).
        for i in range(nH):
            Q[2*k,nS*4+i] = (1.0 / nH) * (alphaTSHpreD / wsS[k])
    
        # Diversion.  Go directly to the post-Diversion shelter that represents this same one.
        Q[2*k,nS*2+2*k] = alphaDiv / wsS[k]
        
        # Dwell rate.  Sum to avoid floating point errors.
        Q[2*k,2*k] = np.sum(-Q[2*k,:])  # Dwell (can be calculated but use sum to avoid precision errors).
    
        # Transitions for absent but returning to shelter state.
        Q[2*k+1,2*k] = alphaR / wsS[k] * (1-alphaTSHpreD-alphaDiv-alphaXS) / (1+alphaR)
        Q[2*k+1,2*k+1] = -alphaR / wsS[k] * (1-alphaTSHpreD-alphaDiv-alphaXS) / (1+alphaR)
    
    # Shelter Programs (postDiversion)
    for k in range(nS):
        # Absent but returning to shelter.    
        Q[nS*2+2*k,nS*2+2*k+1] = alphaR / wsS[nS+k] * (1-alphaTSHpstD-alphaXS) / (1+alphaR)
    
        # Transition to other shelter.  Split between higher and lower shelter numbers.
        Q[nS*2+2*k,nS*2+( (2*k+2) % (2*nS) )] = 0.5 / wsS[nS+k] * (1-alphaTSHpstD-alphaXS) / (1+alphaR)  # Higher
        Q[nS*2+2*k,nS*2+( (2*k+2*(nS-1)) % (2*nS) )] = 0.5 / wsS[nS+k] * (1-alphaTSHpstD-alphaXS) / (1+alphaR)  # Higher
        
        # System exit.
        Q[nS*2+2*k,nS*4+nH] = alphaXS / wsS[nS+k]  
    
        # Transition to housing.  Split equally across all housing programs (for now).
        for i in range(nH):
            Q[nS*2+2*k,nS*4+i] = (1.0 / nH) * (alphaTSHpstD / wsS[nS+k])
        
        # Dwell rate.  Sum to avoid floating point errors.
        Q[nS*2+2*k,nS*2+2*k] = np.sum(-Q[nS*2+2*k,:])  # Dwell (use sum to avoid precision errors).
    
        # Transitions for absent but returning to shelter state.
        Q[nS*2+2*k+1,nS*2+2*k] = alphaR / wsS[nS+k] * (1-alphaTSHpstD-alphaXS) / (1+alphaR)
        Q[nS*2+2*k+1,nS*2+2*k+1] = -alphaR / wsS[nS+k] * (1-alphaTSHpstD-alphaXS) / (1+alphaR)
    
    
    # Housing Programs
    for i in range(nH):
        # Transitions back to post diversion shelter absence state.
        for k in range(nS):
            Q[4*nS+i,nS*2+2*k+1] = (1.0 / nS) * (alphaTHS / wsH[i])
    
        # Exit from system.
        Q[4*nS+i,nS*4+nH] = alphaXH / wsH[i]
    
        # Dwell rate.
        Q[4*nS+i,4*nS+i] = np.sum(-Q[4*nS+i,:])
    
    
    
    R = Q[:-1,:-1]
    loadEst = np.linalg.inv(-R).T @ lam
    W = np.linalg.inv(-R) @ np.ones((nS*4+nH,1)) 
    
    if converged:
        # Final system parameters.
        return Q,W[:,0],loadEst
    else:
        # MSE between actual and target program loads.
        return np.sum( ((loadEst[:2*nS:2] + loadEst[2*nS:4*nS:2]) - loadTrg[:nS]) ** 2 ) + np.sum( (loadEst[nS*4:] - loadTrg[nS:]) ** 2 )

In [9]:
from scipy.optimize import minimize

# Parameters specfied by the community.
nS = 4  # Number of shelters.
nH = 10  # Number of housing programs.
wAvgS = 120  # Average shelter episode (days).
wAvgH = 300  # Average housing episode (days).
alphaDiv = 0.5 # Percentage of shelter users that receive diversion.
alphaTSHpreD = 0.07  # Percentage of shelter users housed (pre-diversion).
alphaTSHpstD = 0.16  # Percentage of shelter users housed (post-diversion).
loadTrg = [ 
    450, 250, 150, 100 # Average shelter episode load.
]
loadTrg += [ 70 ]*10  # Average housing program episode load.


# Initial guesses for optimized parameters.
x0 =  [ 1 ]*nS*2 # Shelter dwell ratios (pre/pst diversion).
x0 += [ 1 ]*nH # Housing dwell ratios.
x0 += [ 1 ]*nS*2 # Shelter state arrival rates.
x0 += [ 0.75 ] # alphaR
x0 += [ 0.01 ] # alphaTHS
x0 += [ 0.1 ] # alphaXS
x0 += [ 0.3 ] # alphaXH

# Optimization bounds.
bounds =  [ (1e-2,4) ]*(nS*2+nH) # Dwell ratios
bounds += [ (1e-4,10) ]*nS*2  # Shelter arrival rates
bounds += [ (1e-2,4) ] # alphaR 
bounds += [ (1e-2,0.9) ] # alphaTHS 
bounds += [ (1e-2,0.9) ] # alphaXS
bounds += [ (1e-2,0.9) ] # alphaXH


res = minimize(
    calc_q_shelter_housing,
    x0,
    bounds=bounds,
    options={'maxiter':10000,'adaptive':True},
    args=(False,nS,nH,wAvgS,wAvgH,alphaDiv,alphaTSHpreD,alphaTSHpstD,loadTrg),
    method='Nelder-Mead'
)

print("Optimal arguments:", res.x)
print("Minimum value:", res.fun)

Optimal arguments: [1.1295386  0.87233708 0.36835318 0.56786089 0.86292007 0.52793086 0.42911779 0.4277385  1.17994147 1.17992402 1.17995764 1.1799605  1.17995821 1.17994519 1.17992105 1.1799522  1.17995985 1.17996606
 0.697937   1.01869578 0.38805485 0.80968059 0.62023411 0.75377281 0.26004729 0.22661473 1.03655343 0.0205099  0.29591899 0.67120472]
Minimum value: 5.133618003453773e-05


### Summarize Model

In [10]:
Q,W,loadThry = calc_q_shelter_housing(res.x,True,nS,nH,wAvgS,wAvgH,alphaDiv,alphaTSHpreD,alphaTSHpstD,loadTrg)

for k in range(2*nS):
    print(f'Shelter {k} > InPrg: {-1/Q[2*k,2*k]:.3f} days, Return: {1/Q[2*k,2*k+1]:.3f} days, SysExit: {W[2*k]:.3f} days')

print('')

for k in range(nS):
    print(f'Load (Shelters {k} & {k+nS}): {loadThry[2*k]+loadThry[2*k+2*nS]:.1f}')

print('')

for i in range(nH):
    print(f'Housing {i} > InPrg: {-1/Q[4*nS+i,4*nS+i]:.3f} days, SysExit: {W[4*nS+i]:.3f} days, Load: {loadThry[4*nS+i]:.1f}')


Shelter 0 > InPrg: 135.545 days, Return: 1986.183 days, SysExit: 675.912 days
Shelter 1 > InPrg: 104.680 days, Return: 1533.919 days, SysExit: 560.012 days
Shelter 2 > InPrg: 44.202 days, Return: 647.713 days, SysExit: 402.581 days
Shelter 3 > InPrg: 68.143 days, Return: 998.528 days, SysExit: 463.740 days
Shelter 4 > InPrg: 103.550 days, Return: 373.932 days, SysExit: 575.330 days
Shelter 5 > InPrg: 63.352 days, Return: 228.770 days, SysExit: 479.202 days
Shelter 6 > InPrg: 51.494 days, Return: 185.951 days, SysExit: 431.344 days
Shelter 7 > InPrg: 51.329 days, Return: 185.353 days, SysExit: 445.947 days

Load (Shelters 0 & 4): 450.0
Load (Shelters 1 & 5): 250.0
Load (Shelters 2 & 6): 150.0
Load (Shelters 3 & 7): 100.0

Housing 0 > InPrg: 511.746 days, SysExit: 533.286 days, Load: 70.0
Housing 1 > InPrg: 511.739 days, SysExit: 533.279 days, Load: 70.0
Housing 2 > InPrg: 511.753 days, SysExit: 533.293 days, Load: 70.0
Housing 3 > InPrg: 511.755 days, SysExit: 533.295 days, Load: 70.0
H

In [11]:
zA = np.zeros(nS*4+nH)
zA[:nS*2] = res.x[nS*2+nH:nS*2+nH+nS*2]
print(f'Arrival Rates: {zA}')

Arrival Rates: [0.697937   1.01869578 0.38805485 0.80968059 0.62023411 0.75377281 0.26004729 0.22661473 0.         0.         0.         0.         0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.         0.         0.        ]


## Markov Model Simulation
---

### Generate Episodes

In [12]:
# User defined system parameters.
leftTimeBuffer = 10*365 # Days before epoch to allow the system to stabilize.
rightTimeBuffer = 10*365 # Days after epoch to avoid right censoring of stats.
recEpoch = 365*5 # Days in the epoch of interest.
nDays = leftTimeBuffer + recEpoch + rightTimeBuffer  # Total simulated days.

In [13]:
nState = len(Q)-1

# Matrix of vectors that only contain transition rates, not dwell rates.
qTrans = Q[ ~np.eye(len(Q),dtype='bool') ][:-(len(Q)-1)].reshape(len(Q)-1,len(Q)-1)
# Vector of only dwell rates, not including the absorbing system exit state.
qDwell = -Q[np.eye(len(Q),dtype='bool')][:-1]

# Cumulative probability vectors used to determine the transition to the next state.
pTrn = ( qTrans / qDwell[:,np.newaxis] ).cumsum(axis=1)
# Indicies of the transition states.
pIdx = np.tile(np.arange(len(Q)),(len(Q),1)) [ ~np.eye(len(Q),dtype='bool') ] [:-(len(Q)-1)].reshape(len(Q)-1,len(Q)-1)

# Absorbing exit state.
exitState = len(Q)-1

In [14]:
mxPeople = 100000 # Max possible people to avoid dynamic array resizing.

In [15]:
arrivals = np.random.poisson(lam=zA[:,np.newaxis],size=(nS*4+nH,nDays))

In [16]:
def pctErrStr(a,b):
    return f'{100*np.abs(a-b)/(a+1e-12):.2f}\\%'

In [17]:
print('Arrival Rate Check:')
zAMeas = np.mean(arrivals,axis=1)
for iState in range(nState):
    print(f'  State {iState} > zThry: {zA[iState]:.3f}, zMeas: {zAMeas[iState]:.3f}, PctErr: {pctErrStr(zA[iState],zAMeas[iState])}%')

Arrival Rate Check:
  State 0 > zThry: 0.698, zMeas: 0.689, PctErr: 1.24\%%
  State 1 > zThry: 1.019, zMeas: 0.999, PctErr: 1.91\%%
  State 2 > zThry: 0.388, zMeas: 0.378, PctErr: 2.71\%%
  State 3 > zThry: 0.810, zMeas: 0.804, PctErr: 0.69\%%
  State 4 > zThry: 0.620, zMeas: 0.611, PctErr: 1.55\%%
  State 5 > zThry: 0.754, zMeas: 0.757, PctErr: 0.48\%%
  State 6 > zThry: 0.260, zMeas: 0.256, PctErr: 1.60\%%
  State 7 > zThry: 0.227, zMeas: 0.230, PctErr: 1.31\%%
  State 8 > zThry: 0.000, zMeas: 0.000, PctErr: 0.00\%%
  State 9 > zThry: 0.000, zMeas: 0.000, PctErr: 0.00\%%
  State 10 > zThry: 0.000, zMeas: 0.000, PctErr: 0.00\%%
  State 11 > zThry: 0.000, zMeas: 0.000, PctErr: 0.00\%%
  State 12 > zThry: 0.000, zMeas: 0.000, PctErr: 0.00\%%
  State 13 > zThry: 0.000, zMeas: 0.000, PctErr: 0.00\%%
  State 14 > zThry: 0.000, zMeas: 0.000, PctErr: 0.00\%%
  State 15 > zThry: 0.000, zMeas: 0.000, PctErr: 0.00\%%
  State 16 > zThry: 0.000, zMeas: 0.000, PctErr: 0.00\%%
  State 17 > zThry: 0

In [18]:
# Each person occupies two rows: state number and state exit time.
# - State entry time for the first state is stored in tRecStart.
# - Entry times for subsequent states are the previous states' exit times.
# - The exit state will have no exit time.
recs = np.zeros((mxPeople*2,nDays))

edRec = np.zeros(mxPeople,dtype='int') # Column index of last record.
tRecStart = np.zeros(mxPeople) # Record start time.

# Useful masks and arrays for indexing.
allRows = np.arange(mxPeople*2)  # Necessary since using : sometimes causes huge arrays to be built
stateRows = allRows[0::2]
timeRows = allRows[1::2]

recs[ stateRows ] = -1
active = np.array([ False ] * mxPeople )

In [19]:
freeRecIdx = 0

In [20]:
# Time resolution is in days.
debug = False
for t in tqdm(range(nDays)):

    if debug:
        print(f'\n------------- t = {t} --------------')

    # Identify active users whose dwell times have expired.
    activeUsers = np.where(active)[0]
    expUsers = activeUsers[ np.where( recs[ timeRows[active], edRec[active] ] <= t )[0] ]
    nExp = len(expUsers)

    expCurStates = recs[ stateRows[expUsers], edRec[expUsers] ].astype(int) # Expired user current state numbers.
    
    # Randomly determine the transition state.
    u = np.random.uniform( size = nExp )
    # Select the transition state indicies from pIdx and then determine which one is chosen using pTrn.
    nxtStates = pIdx[expCurStates] [ np.arange(nExp), np.argmin( pTrn[expCurStates] < u[:,np.newaxis],axis=1) ]

    nxtDwellRates = Q[nxtStates,nxtStates] # Transition state dwell rates.
    nxtDwellRates[ nxtDwellRates == 0 ] = -1 # Dummy dwell rate for the exit state.  

    # Transition state dwell times.
    tNxtDwell = np.random.exponential(scale=-1/nxtDwellRates) 
    tNxtDwell[ tNxtDwell < 1 ] = 1 # Ensure all episodes are at least one day long.
    tNxtDwell += recs[ timeRows[expUsers], edRec[expUsers] ]  
    
    tNxtDwell[ nxtDwellRates == -1 ] = 0 # Set the dwell time to zero for exit states.
    
    # Create and assign the new record entries for expired users.
    recs[ stateRows[expUsers], edRec[expUsers]+1 ] = nxtStates
    recs[ timeRows[expUsers], edRec[expUsers]+1 ] = tNxtDwell

    edRec[expUsers] += 1  # Increment record counter for the new entries.

    # Deactivate any users who have exited the system.
    exit = recs[ stateRows, edRec ] == exitState
    active[exit] = False    

    if debug:
        for iD in expUsers:
            print(f' ExpUser {iD} > NxtState: {recs[stateRows[iD],edRec[iD]]}',end='')
            if recs[stateRows[iD],edRec[iD]] == exitState:
                print(f' (EXIT)')
            else:
                print('')
    
    # Check for new arrivals.
    for iState in range(nState):
        if arrivals[iState,t] > 0:

            # Record record start time.
            nArr = arrivals[iState,t] 
            tRecStart[ freeRecIdx : freeRecIdx + nArr ] = t
        
            # Create new arrival record index mask and update new record counter.
            arrived = np.zeros(mxPeople,dtype='bool')
            arrived[ freeRecIdx : freeRecIdx + nArr ] = True
            freeRecIdx += nArr
    
            active[arrived] = True # Designate the new users as active.
    
            # Init the new state to in-program.
            recs[ stateRows[ arrived ], 0 ] = iState
            
            # Determine dwell times for each new arrival.
            initStates = recs[stateRows[arrived], 0 ].astype('int')  
            initDwellRates = Q[initStates,initStates]
            tDwell = np.random.exponential(scale=-1/initDwellRates)
            tDwell[ tDwell < 1 ] = 1
            tDwell += t 
        
            # Assign dwell times.
            recs[ timeRows[arrived], 0 ] = tDwell
    
            if debug:
                for iD in np.where(arrived)[0]:
                    print(f' NewUser {iD} > State: {iState}')
    


  0%|          | 0/9125 [00:00<?, ?it/s]

In [21]:
# Remove extra episode array storage.
edRec = edRec[:freeRecIdx]
tRecStart = tRecStart[:freeRecIdx]
recs = recs[:freeRecIdx*2,:edRec.max()+2]
nPeople = freeRecIdx
allRows = np.arange(nPeople*2)
stateRows = allRows[0::2]
timeRows = allRows[1::2]

### Verify Episode Data

In [22]:
# Only check shelter states for mean time to absorption since housing is always an intermediate state.
print('Mean Time to Absorption:')
for iState in range(nS*2):
    strtdInPrg = recs[stateRows][:,0] == iState
    fullRecs = np.max( recs[stateRows][strtdInPrg] == exitState, axis=1 )
    notCensored = ( tRecStart[strtdInPrg][fullRecs] < leftTimeBuffer )
    recLen = np.max( recs[timeRows][strtdInPrg][fullRecs][notCensored], axis=1 ) - tRecStart[strtdInPrg][fullRecs][notCensored]

    print(f'  State {iState} > Theory: {W[iState]:.3f}, Sim: {np.mean(recLen):.3f} (PctErr: {pctErrStr(W[iState],np.mean(recLen))})')
    

Mean Time to Absorption:
  State 0 > Theory: 675.912, Sim: 621.172 (PctErr: 8.10\%)
  State 1 > Theory: 2662.094, Sim: 2340.571 (PctErr: 12.08\%)
  State 2 > Theory: 560.012, Sim: 563.927 (PctErr: 0.70\%)
  State 3 > Theory: 2093.931, Sim: 2001.985 (PctErr: 4.39\%)
  State 4 > Theory: 402.581, Sim: 386.700 (PctErr: 3.94\%)
  State 5 > Theory: 1050.294, Sim: 1059.119 (PctErr: 0.84\%)
  State 6 > Theory: 463.740, Sim: 427.802 (PctErr: 7.75\%)
  State 7 > Theory: 1462.268, Sim: 1406.832 (PctErr: 3.79\%)


In [23]:
loadSim = []
for iState in range(nState):

    iStay, jStay = np.where(recs[stateRows] == iState)
    nEpis = len(iStay)
    tEpi = np.zeros((nEpis, 2),dtype='int')

    # First stays
    isFirst = (jStay == 0)
    iFirstStay = iStay[isFirst]
    tEpi[isFirst, 0] = (tRecStart[iFirstStay]).astype('int')
    tEpi[isFirst, 1] = (recs[timeRows][iFirstStay, 0]).astype('int')+1

    # Subsequent stays
    isSubsqnt = ~isFirst
    iSub = iStay[isSubsqnt]
    jSub = jStay[isSubsqnt]
    tEpi[isSubsqnt, 0] = (recs[timeRows][iSub, jSub - 1]).astype('int')
    tEpi[isSubsqnt, 1] = (recs[timeRows][iSub, jSub]).astype('int')+1
    
    load = np.zeros(nDays,dtype='int')  # 1 day resolution
    
    np.add.at( load, tEpi[:,0], 1 )
    inBounds = tEpi[:,1] < nDays
    np.add.at( load, tEpi[inBounds,1], -1 )
    
    load = load.cumsum()
    loadSim += [ np.mean(load[leftTimeBuffer:]) ]

In [24]:
print('Shelter Program Load')
for n in range(nS):
    loadS = loadSim[2*n] + loadSim[2*(n+nS)]
    loadT = loadThry[2*n] + loadThry[2*(n+nS)]
    print(f'{n} & {loadS:.0f} & {loadT:.0f} & {pctErrStr(loadS,loadT)} \\\\')

Shelter Program Load
0 & 425 & 450 & 5.83\% \\
1 & 244 & 250 & 2.40\% \\
2 & 149 & 150 & 0.84\% \\
3 & 101 & 100 & 0.71\% \\


In [25]:
print('Housing Program Load')
for i in range(nH):
    loadS = loadSim[4*nS+i]
    loadT = loadThry[4*nS+i]
    print(f'{i} & {loadS:.0f} & {loadT:.0f} & {pctErrStr(loadS,loadT)} \\\\')

Housing Program Load
0 & 72 & 70 & 3.29\% \\
1 & 60 & 70 & 17.12\% \\
2 & 67 & 70 & 4.19\% \\
3 & 67 & 70 & 4.02\% \\
4 & 63 & 70 & 10.57\% \\
5 & 64 & 70 & 10.00\% \\
6 & 75 & 70 & 6.39\% \\
7 & 64 & 70 & 9.10\% \\
8 & 66 & 70 & 6.66\% \\
9 & 70 & 70 & 0.23\% \\


## Collapse Diversion Data and Add Diversion Event
---
- A zero time diversion event is inserted on the first day a person enters a post-div shelter.

In [26]:
divState = nS*4+nH+1 # State number for the zero time diversion event.
divShlts = np.arange(0,nS*2) + nS*2 # Post-diversion shelter state numbers
isDivShlt = np.isin(recs[stateRows],divShlts) # Boolean mask of recs indicating where the diversion shelter entries are.
hasDiv = isDivShlt.sum(axis=1) > 0 # The rows that include post-div shelters.

In [27]:
repIdx = np.argmax(isDivShlt,axis=1) # Array of the indices of the first post-div shelter entry. Equals zero if no post-div shelter states.
repIdx[~hasDiv] = edRec[~hasDiv]+1 # Replace the zero entries with an off record index (so repeating entry does nothing).

In [28]:
r_grid, c_grid = np.indices(recs.shape) # 2d arrays of row and column indices.

# Modify column indices to repeat the diversion entries.
source_cols = np.where(c_grid <= np.repeat(repIdx,2)[:, None], c_grid, c_grid - 1)

In [29]:
recs = recs[r_grid, source_cols]  # Update with repeated diversion entries.
recs[stateRows[hasDiv],repIdx[hasDiv]] = divState  # Assign diversion event the new state number.

In [30]:
# Collapse pre/post div shelters into a single set of state numbers.
for pstDivShlt in divShlts:
    recs[ recs == pstDivShlt ] = pstDivShlt - nS*2

In [31]:
# The diversion event needs the same dwell time as the previous event.
jDiv = np.argmax(recs[stateRows[hasDiv]] == divState,axis=1)
recs[ timeRows[hasDiv] ,jDiv ] = recs[ timeRows[hasDiv] ,jDiv-1 ]

## Create Full System Records
---

In [32]:
startYear = 2025 # The year where the epoch of interest begins.
epochStart = pd.to_datetime(f'{startYear}-1-1')

In [33]:
recsMkv = copy.deepcopy(recs)

#### Remove Unrealistically Long Records

In [34]:
tooLongDuration = 365 * 10

In [35]:
lenRecs = np.max(recsMkv[timeRows],axis=1) - tRecStart
notTooLong = lenRecs < tooLongDuration
print(f'{np.sum(notTooLong)} / {nPeople} not too long.')

recsMkv = recsMkv[ np.repeat(notTooLong,2) ]
tRecStart = tRecStart[ notTooLong ]
nPeople = np.sum(notTooLong)

39762 / 43103 not too long.


In [36]:
allRows = np.arange(nPeople*2)
stateRows = allRows[0::2]
timeRows = allRows[1::2]

#### Only Consider Records in Epoch of Interest

In [37]:
tRecEnd = np.max(recsMkv[timeRows],axis=1)
tEpochStart = leftTimeBuffer
tEpochEnd = tEpochStart + recEpoch

epochOverlap = (tRecStart >= tEpochStart) & (tRecStart <= tEpochEnd)
epochOverlap |= (tRecEnd >= tEpochStart) & (tRecEnd <= tEpochEnd)
epochOverlap |= (tRecStart < tEpochStart) & (tRecEnd > tEpochEnd)

In [38]:
print(f'{np.sum(epochOverlap)} / {nPeople} overlap with epoch of interest.')

recsMkv = recsMkv[ np.repeat(epochOverlap,2) ]
tRecStart = tRecStart[ epochOverlap ]
nPeople = np.sum(epochOverlap)

12093 / 39762 overlap with epoch of interest.


In [39]:
allRows = np.arange(nPeople*2)
stateRows = allRows[0::2]
timeRows = allRows[1::2]

In [40]:
recs = []
for iR in range(nPeople):
    recs += [ {} ]
    recs[iR]['Start'] = float( tRecStart[iR] )
    recs[iR]['End'] = float( np.max(recsMkv[timeRows[iR]]   ) ) 

### Names
---
- For now, every record is unique and perfectly linked.
- Could consider splitting these records by service provider and giving misspelled names like in the synthetic record linkage data set.

In [41]:
from names_dataset import NameDataset, NameWrapper

In [42]:
nd = NameDataset()

In [43]:
firstNames = list( nd.first_names.keys() )
lastNames = list( nd.last_names.keys() )

In [44]:
firstNames = [ s for s in firstNames if (s.isascii() and ' ' not in s and 'CA' in list( nd.first_names[s]['country'].keys() ) ) ]
lastNames = [ s for s in lastNames if (s.isascii() and ' ' not in s and 'CA' in list( nd.last_names[s]['country'].keys() ) ) ]

In [45]:
for iR in range(len(recs)):
    recs[iR]['FirstName'] = firstNames[ np.random.randint(len(firstNames)) ]
    recs[iR]['LastName'] = lastNames[ np.random.randint(len(lastNames)) ]    

### Age
---

In [46]:
age = read_pickle(procDirStr + 'CHF-Ages.pkl')

In [47]:
ageCdf = ( age['H'] / age['H'].sum() ).cumsum()
dobYear = startYear - age['X']

In [48]:
for iR in range(len(recs)):
    recs[iR]['Dob'] = pd.to_datetime(np.random.randint(1,60*60*24*365),unit='s').normalize()
    recs[iR]['Dob'] = recs[iR]['Dob'].replace(year=int( dobYear[ ageCdf >= np.random.uniform() ][0] ))

### Episode Records
---

In [49]:
prgIds = {}
prgIds['ShelterEpis'] = np.arange(0,nS*2,2,dtype='int')
prgIds['HousingEpis'] =  np.arange(nS*4,nS*4+nH,dtype='int')
prgIds['Diversion'] = np.array([ divState ])
exitId = nS*4+nH

In [50]:
for iR in tqdm( range(nPeople) ):
    nEpi = np.sum(recsMkv[ stateRows[iR] ] != -1 )
    epis = np.zeros((nEpi,3),dtype='int')
    
    epis[0,0] = recsMkv[ stateRows[iR] ][0]
    epis[0,1] = tRecStart[iR]
    epis[0,2] = recsMkv[ timeRows[iR] ][0] 
    
    epis[1:nEpi,0] = recsMkv[ stateRows[iR] ][1:nEpi]
    epis[1:nEpi,1] = recsMkv[ timeRows[iR] ][0:nEpi-1]
    epis[1:nEpi,2] = recsMkv[ timeRows[iR] ][1:nEpi]

    for epiType in [ 'ShelterEpis', 'HousingEpis', 'Diversion' ]:
        prgIdx = np.isin(epis[:,0],prgIds[epiType])
        epiDict = {}
        if epiType == 'ShelterEpis':
            epiDict['ProgramId'] = [ int(x/2) for x in epis[prgIdx,0] ]
        elif epiType == 'HousingEpis':
            epiDict['ProgramId'] = [ int(x-nS*4) for x in epis[prgIdx,0] ]
        elif epiType == 'Diversion':
            epiDict['ProgramId'] = [ int(x-nS*4-nH) for x in epis[prgIdx,0] ]
        epiDict['StartDate'] = list( epochStart + pd.to_timedelta(epis[prgIdx,1]-leftTimeBuffer,unit='d') )
        epiDict['EndDate'] = list( epochStart + pd.to_timedelta(epis[prgIdx,2]-leftTimeBuffer,unit='d') )    
    
        recs[iR][epiType] = epiDict


  0%|          | 0/12093 [00:00<?, ?it/s]

## Save Data Set
---

In [51]:
dataSet = {}
dataSet['StartYear'] = startYear
dataSet['RecEpoch'] = recEpoch
dataSet['NShelters'] = nS
dataSet['NHousing'] = nH
dataSet['DivState'] = divState
dataSet['Records'] = recs

In [52]:
dump_pickle(dataSet,procDirStr + 'SHHAd-System-Level-2026_11_03.pkl')

## Create Master Table
---

In [53]:
housePrgNames = pd.read_csv(procDirStr + 'Names-HousingPrograms.csv')['Name']
shelterNames = pd.read_csv(procDirStr + 'Names-EmergencyShelters.csv')['Name']
triagePrgNames = pd.read_csv(procDirStr + 'Names-ReferralAgencies.csv')['Name']

In [54]:
len(triagePrgNames)

19

In [55]:
tblDct = {
    'PersonId': {},
    'ServiceType': {},
    'StartDate': {},
    'EndDate': {},
    'FirstName': {},
    'LastName': {},
    'DOB': {},
    'ProviderId': {},
    'ProviderName': {}
}

In [56]:
idx = 0

In [57]:
serviceTypes = {
    'ShelterEpis': 'Shelter', 
    'HousingEpis': 'Housing',
    'Diversion': 'Diversion' 
}
prgIdOffset = {
    'ShelterEpis': 2000, 
    'HousingEpis': 3000,
    'Diversion': 1000 
}
prgNames = {
    'ShelterEpis': shelterNames, 
    'HousingEpis': housePrgNames,
    'Diversion': triagePrgNames 
}

In [58]:
for iR in range(len(recs)):

    for epiType in serviceTypes.keys():
        for iE in range( len(recs[iR][epiType]['ProgramId']) ): 
            tblDct['PersonId'].update({ idx: iR })
            tblDct['ServiceType'].update({ idx: serviceTypes[epiType] })
            
            tblDct['StartDate'].update({ idx: recs[iR][epiType]['StartDate'][iE] })
            tblDct['EndDate'].update({ idx: recs[iR][epiType]['EndDate'][iE] })
            
            tblDct['FirstName'].update({ idx: recs[iR]['FirstName'] })
            tblDct['LastName'].update({ idx: recs[iR]['LastName'] })
            tblDct['DOB'].update({ idx: recs[iR]['Dob'] })
            
            tblDct['ProviderId'].update({ idx: int(recs[iR][epiType]['ProgramId'][iE]) + prgIdOffset[epiType] })
            tblDct['ProviderName'].update({ idx: prgNames[epiType][ recs[iR][epiType]['ProgramId'][iE] ] })
            idx += 1  

In [59]:
tbl = pd.DataFrame(tblDct)

In [60]:
tbl

,PersonId,ServiceType,StartDate,EndDate,FirstName,LastName,DOB,ProviderId,ProviderName
0,0,Shelter,2017-05-20,2017-07-05,Linzie,Rwandzy,1965-04-25,2001,Common Roof Emergency Shelter
1,0,Shelter,2017-07-05,2017-10-03,Linzie,Rwandzy,1965-04-25,2001,Common Roof Emergency Shelter
2,0,Shelter,2017-10-03,2018-03-09,Linzie,Rwandzy,1965-04-25,2000,Keystone Emergency Centre
3,0,Shelter,2020-04-03,2020-04-06,Linzie,Rwandzy,1965-04-25,2000,Keystone Emergency Centre
4,0,Shelter,2020-04-20,2020-10-10,Linzie,Rwandzy,1965-04-25,2000,Keystone Emergency Centre
...,...,...,...,...,...,...,...,...,...
41530,12092,Shelter,2030-06-13,2030-08-21,Brahimo,Vooys,1992-11-26,2001,Common Roof Emergency Shelter
41531,12092,Shelter,2032-05-17,2032-08-27,Brahimo,Vooys,1992-11-26,2001,Common Roof Emergency Shelter
41532,12092,Shelter,2032-10-23,2032-12-08,Brahimo,Vooys,1992-11-26,2001,Common Roof Emergency Shelter
41533,12092,Shelter,2033-04-22,2033-07-17,Brahimo,Vooys,1992-11-26,2001,Common Roof Emergency Shelter


In [61]:
tbl.to_parquet(procDirStr + 'SHHAd-System-Level-2026_11_03.parquet')

In [62]:
tbl.to_csv(procDirStr + 'SHHAd-System-Level-2026_11_03.csv')